# Chiron APM - Insolation Selection Logic Research

## Objective
Analyze 90 days of insolation data across the fleet to:
1. Quantify onsite vs satellite agreement/disagreement patterns
2. Identify failure modes (sensor stuck, dirty, satellite misses, etc.)
3. Find regional/seasonal/site-specific patterns
4. Develop and validate improved selection logic

## Key Metrics
- **WA PR (Weather-Adjusted Performance Ratio)**: Production / WA_Expected
- **Insolation Gap**: (actual / expected) - 1
- **Onsite vs Satellite Ratio**: onsite_insolation / satellite_insolation

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from pathlib import Path
import sys
import warnings
warnings.filterwarnings('ignore')

# Set up paths for imports
CHIRON_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(CHIRON_ROOT))
sys.path.insert(0, str(CHIRON_ROOT.parent))

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f"Working from: {Path.cwd()}")
print(f"CHIRON_ROOT: {CHIRON_ROOT}")

In [ ]:
# Import Chiron data service
from CHIRON_MONITORING.config import Config
from CHIRON_MONITORING.data import SnowflakeConnector

# Load config
config_path = CHIRON_ROOT / "config.json"
config = Config(config_file=str(config_path))

# Connect to Snowflake
connector = SnowflakeConnector(**config.snowflake.connection_params)
print("Connected to Snowflake successfully!")

## 1. Pull 90 Days of Fleet Data

Get all insolation sources for Post-FC sites (operational sites with forecasts).

In [ ]:
# Define date range
end_date = datetime.now().strftime('%Y-%m-%d')
start_date = (datetime.now() - timedelta(days=90)).strftime('%Y-%m-%d')

print(f"Analysis period: {start_date} to {end_date}")

In [ ]:
# Query all daily data with all insolation sources
query_daily = f"""
SELECT 
    d.SITEID,
    d.SITENAME,
    d.MEASUREMENTTIME::DATE as DATE,
    d.STAGE,
    
    -- Production data
    d.PRODUCTION,
    d.EXPECTED_PRODUCTION,
    d.PRODUCTION_SOURCE,
    d.METER_ENERGY,
    d.INV_TOTAL_ENERGY,
    
    -- All raw insolation sources
    d.INSOLATION_POA as ONSITE_POA,
    d.INSOLATION_GHI as ONSITE_GHI,
    d.INSOLATION_GHI_SOLCAST as SATELLITE_GHI,
    d.INSOLATION_POA_SOLCAST as SATELLITE_POA,
    
    -- Expected insolation
    d.EXPECTED_INSOLATION_POA,
    d.EXPECTED_INSOLATION_GHI,
    
    -- View's current selection
    d.INSOLATION as VIEW_INSOLATION,
    d.INSOLATION_SOURCE as VIEW_SOURCE,
    d.INSOLATION_GAP as VIEW_GAP,
    d.WA_PERFORMANCE_RATIO as VIEW_WA_PR,
    d.PERFORMANCE_RATIO as VIEW_PR,
    
    -- Site metadata
    sm.IRRADIANCE_TYPE,
    sm.SIZE_KW_DC,
    sm.SIZE_KW_AC,
    sm.STATE,
    sm.LATITUDE,
    sm.LONGITUDE
    
FROM MEI_ASSET_MGMT_DB.PERFORMANCE.DAILY_DATA_LIVE d
JOIN MEI_ASSET_MGMT_DB.PUBLIC.SITE_MASTER sm ON d.SITEID = sm.SITE_ID
WHERE d.DATA_TYPE = 'current'
  AND d.MEASUREMENTTIME >= '{start_date}'
  AND d.MEASUREMENTTIME <= '{end_date}'
  AND d.STAGE = 'Post-FC'
ORDER BY d.SITEID, d.MEASUREMENTTIME
"""

print("Executing query...")
result = connector.execute_query(query_daily)
df = pd.DataFrame(result)
df['DATE'] = pd.to_datetime(df['DATE'])

print(f"\nLoaded {len(df):,} site-days")
print(f"Sites: {df['SITEID'].nunique()}")
print(f"Date range: {df['DATE'].min()} to {df['DATE'].max()}")

In [ ]:
# Quick data overview
print("=" * 60)
print("DATA OVERVIEW")
print("=" * 60)
print(f"\nTotal site-days: {len(df):,}")
print(f"Unique sites: {df['SITEID'].nunique()}")
print(f"\nIrradiance Type Distribution:")
print(df['IRRADIANCE_TYPE'].value_counts())
print(f"\nView Source Distribution:")
print(df['VIEW_SOURCE'].value_counts())

## 2. Compute Derived Metrics

Calculate key ratios and flags for analysis.

In [ ]:
# Make a copy for analysis
data = df.copy()

# Compute onsite vs satellite ratios
# Use the preferred onsite based on IRRADIANCE_TYPE
data['PREFERRED_ONSITE'] = np.where(
    data['IRRADIANCE_TYPE'] == 2,  # POA preferred
    data['ONSITE_POA'],
    data['ONSITE_GHI']
)

# Ratio: onsite / satellite_ghi
data['ONSITE_SAT_RATIO'] = data['PREFERRED_ONSITE'] / data['SATELLITE_GHI'].replace(0, np.nan)

# Ratio: POA onsite / satellite_ghi (POA should be higher than GHI)
data['POA_SAT_RATIO'] = data['ONSITE_POA'] / data['SATELLITE_GHI'].replace(0, np.nan)
data['GHI_SAT_RATIO'] = data['ONSITE_GHI'] / data['SATELLITE_GHI'].replace(0, np.nan)

# Percentage difference from satellite
data['ONSITE_SAT_DIFF_PCT'] = (data['PREFERRED_ONSITE'] - data['SATELLITE_GHI']) / data['SATELLITE_GHI'].replace(0, np.nan) * 100

# Gap from expected
data['EXPECTED_INSOL'] = np.where(
    data['IRRADIANCE_TYPE'] == 2,
    data['EXPECTED_INSOLATION_POA'],
    data['EXPECTED_INSOLATION_GHI']
)
data['EXPECTED_GAP_PCT'] = (data['PREFERRED_ONSITE'] - data['EXPECTED_INSOL']) / data['EXPECTED_INSOL'].replace(0, np.nan) * 100

print("Computed metrics:")
print(data[['SITEID', 'DATE', 'PREFERRED_ONSITE', 'SATELLITE_GHI', 'ONSITE_SAT_RATIO', 'ONSITE_SAT_DIFF_PCT']].head(10))

In [ ]:
# Flag problematic readings
MIN_THRESHOLD = 100  # Wh/m2/day minimum

data['ONSITE_ZERO'] = data['PREFERRED_ONSITE'] < MIN_THRESHOLD
data['SATELLITE_ZERO'] = data['SATELLITE_GHI'] < MIN_THRESHOLD

# Flag extreme ratios
data['RATIO_TOO_LOW'] = data['ONSITE_SAT_RATIO'] < 0.2  # Onsite < 20% of satellite
data['RATIO_TOO_HIGH'] = data['ONSITE_SAT_RATIO'] > 5.0  # Onsite > 5x satellite
data['RATIO_MODERATE_LOW'] = (data['ONSITE_SAT_RATIO'] >= 0.2) & (data['ONSITE_SAT_RATIO'] < 0.5)
data['RATIO_MODERATE_HIGH'] = (data['ONSITE_SAT_RATIO'] > 2.0) & (data['ONSITE_SAT_RATIO'] <= 5.0)
data['RATIO_REASONABLE'] = (data['ONSITE_SAT_RATIO'] >= 0.5) & (data['ONSITE_SAT_RATIO'] <= 2.0)

# Flag WA PR issues
data['WA_PR_HIGH'] = data['VIEW_WA_PR'] > 1.2  # >120%
data['WA_PR_LOW'] = data['VIEW_WA_PR'] < 0.5   # <50%
data['WA_PR_OK'] = (data['VIEW_WA_PR'] >= 0.5) & (data['VIEW_WA_PR'] <= 1.2)

print("\n" + "=" * 60)
print("PROBLEM FLAGS SUMMARY")
print("=" * 60)
print(f"\nOnsite Zero (< {MIN_THRESHOLD} Wh/m2): {data['ONSITE_ZERO'].sum():,} ({data['ONSITE_ZERO'].mean()*100:.1f}%)")
print(f"Satellite Zero (< {MIN_THRESHOLD} Wh/m2): {data['SATELLITE_ZERO'].sum():,} ({data['SATELLITE_ZERO'].mean()*100:.1f}%)")
print(f"\nRatio < 0.2 (onsite way lower than satellite): {data['RATIO_TOO_LOW'].sum():,} ({data['RATIO_TOO_LOW'].mean()*100:.1f}%)")
print(f"Ratio > 5.0 (onsite way higher than satellite): {data['RATIO_TOO_HIGH'].sum():,} ({data['RATIO_TOO_HIGH'].mean()*100:.1f}%)")
print(f"Ratio 0.5-2.0 (reasonable agreement): {data['RATIO_REASONABLE'].sum():,} ({data['RATIO_REASONABLE'].mean()*100:.1f}%)")
print(f"\nWA PR > 120%: {data['WA_PR_HIGH'].sum():,} ({data['WA_PR_HIGH'].mean()*100:.1f}%)")
print(f"WA PR < 50%: {data['WA_PR_LOW'].sum():,} ({data['WA_PR_LOW'].mean()*100:.1f}%)")

## 3. Onsite vs Satellite Agreement Analysis

### 3.1 Overall Distribution

In [ ]:
# Filter to rows with valid data for analysis
valid_data = data[~data['ONSITE_ZERO'] & ~data['SATELLITE_ZERO']].copy()
print(f"Rows with valid onsite AND satellite data: {len(valid_data):,} ({len(valid_data)/len(data)*100:.1f}% of total)")

# Summary statistics of ratio
print("\n" + "=" * 60)
print("ONSITE/SATELLITE RATIO STATISTICS")
print("=" * 60)
print(valid_data['ONSITE_SAT_RATIO'].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]))

In [ ]:
# Histogram of onsite/satellite ratio
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full range
ax1 = axes[0]
valid_data['ONSITE_SAT_RATIO'].hist(bins=100, ax=ax1, alpha=0.7, color='steelblue')
ax1.axvline(1.0, color='green', linestyle='--', linewidth=2, label='Perfect Agreement (1.0)')
ax1.axvline(0.2, color='red', linestyle='--', linewidth=1, label='Lower Bound (0.2)')
ax1.axvline(5.0, color='red', linestyle='--', linewidth=1, label='Upper Bound (5.0)')
ax1.set_xlabel('Onsite / Satellite Ratio')
ax1.set_ylabel('Frequency')
ax1.set_title('Full Distribution of Onsite/Satellite Ratio')
ax1.legend()
ax1.set_xlim(0, 10)

# Zoomed to reasonable range
ax2 = axes[1]
reasonable_data = valid_data[(valid_data['ONSITE_SAT_RATIO'] >= 0.5) & (valid_data['ONSITE_SAT_RATIO'] <= 2.0)]
reasonable_data['ONSITE_SAT_RATIO'].hist(bins=50, ax=ax2, alpha=0.7, color='steelblue')
ax2.axvline(1.0, color='green', linestyle='--', linewidth=2, label='Perfect Agreement')
ax2.set_xlabel('Onsite / Satellite Ratio')
ax2.set_ylabel('Frequency')
ax2.set_title(f'Reasonable Range (0.5-2.0): {len(reasonable_data):,} site-days')
ax2.legend()

plt.tight_layout()
plt.savefig('insolation_ratio_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: onsite vs satellite
fig, ax = plt.subplots(figsize=(10, 8))

# Sample for visualization if too many points
plot_data = valid_data.sample(min(5000, len(valid_data)), random_state=42)

scatter = ax.scatter(
    plot_data['SATELLITE_GHI'],
    plot_data['PREFERRED_ONSITE'],
    c=plot_data['VIEW_WA_PR'].clip(0.5, 1.5),
    cmap='RdYlGn',
    alpha=0.5,
    s=10
)

# Perfect agreement line
max_val = max(plot_data['SATELLITE_GHI'].max(), plot_data['PREFERRED_ONSITE'].max())
ax.plot([0, max_val], [0, max_val], 'k--', linewidth=2, label='Perfect Agreement')
ax.plot([0, max_val], [0, max_val * 0.8], 'g--', linewidth=1, alpha=0.5, label='POA typical (1.2x GHI)')
ax.plot([0, max_val], [0, max_val * 1.5], 'b--', linewidth=1, alpha=0.5, label='1.5x line')

ax.set_xlabel('Satellite GHI (Wh/m2/day)')
ax.set_ylabel('Onsite Insolation (Wh/m2/day)')
ax.set_title('Onsite vs Satellite Insolation (color = WA PR)')
plt.colorbar(scatter, label='WA PR')
ax.legend()
ax.set_xlim(0, max_val * 1.1)
ax.set_ylim(0, max_val * 1.1)

plt.tight_layout()
plt.savefig('onsite_vs_satellite_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.2 Failure Mode Identification

In [ ]:
# Categorize each site-day into failure modes
def categorize_failure_mode(row):
    onsite = row['PREFERRED_ONSITE']
    satellite = row['SATELLITE_GHI']
    ratio = row['ONSITE_SAT_RATIO']
    wa_pr = row['VIEW_WA_PR']
    
    # No onsite data
    if onsite < 100 and satellite >= 100:
        return 'ONSITE_DEAD'
    
    # No satellite data
    if satellite < 100 and onsite >= 100:
        return 'SATELLITE_MISSING'
    
    # Both missing
    if onsite < 100 and satellite < 100:
        return 'BOTH_MISSING'
    
    # Onsite stuck/low (likely sensor failure)
    if ratio < 0.2:
        return 'ONSITE_STUCK_LOW'
    
    # Onsite way too high (sensor error or POA vs GHI mismatch)
    if ratio > 5.0:
        return 'ONSITE_EXTREME_HIGH'
    
    # Satellite too low (microclimate miss)
    # Onsite higher but WA PR reasonable - satellite likely underestimated
    if ratio > 1.5 and wa_pr <= 1.0:
        return 'SATELLITE_LOW_MISS'
    
    # Satellite too high (cloud miss)
    # Onsite lower but WA PR reasonable - satellite likely overestimated
    if ratio < 0.7 and wa_pr >= 0.7:
        return 'SATELLITE_HIGH_MISS'
    
    # Suspicious onsite high (sensor error or dirty satellite)
    if ratio > 1.5 and wa_pr > 1.2:
        return 'ONSITE_SUSPICIOUS_HIGH'
    
    # Suspicious onsite low (sensor error)
    if ratio < 0.7 and wa_pr < 0.6:
        return 'ONSITE_SUSPICIOUS_LOW'
    
    # Reasonable agreement
    if 0.7 <= ratio <= 1.5:
        return 'GOOD_AGREEMENT'
    
    return 'MODERATE_DEVIATION'

data['FAILURE_MODE'] = data.apply(categorize_failure_mode, axis=1)

print("\n" + "=" * 60)
print("FAILURE MODE DISTRIBUTION")
print("=" * 60)
failure_counts = data['FAILURE_MODE'].value_counts()
failure_pcts = data['FAILURE_MODE'].value_counts(normalize=True) * 100
failure_summary = pd.DataFrame({'Count': failure_counts, 'Percent': failure_pcts.round(2)})
print(failure_summary)

In [ ]:
# Visualize failure modes
fig, ax = plt.subplots(figsize=(12, 6))

colors = {
    'GOOD_AGREEMENT': 'green',
    'MODERATE_DEVIATION': 'lightgreen',
    'SATELLITE_LOW_MISS': 'skyblue',
    'SATELLITE_HIGH_MISS': 'steelblue',
    'ONSITE_DEAD': 'red',
    'ONSITE_STUCK_LOW': 'darkred',
    'ONSITE_EXTREME_HIGH': 'orange',
    'ONSITE_SUSPICIOUS_HIGH': 'coral',
    'ONSITE_SUSPICIOUS_LOW': 'salmon',
    'SATELLITE_MISSING': 'gray',
    'BOTH_MISSING': 'black'
}

failure_counts = data['FAILURE_MODE'].value_counts()
bars = ax.bar(range(len(failure_counts)), failure_counts.values, 
              color=[colors.get(x, 'gray') for x in failure_counts.index])
ax.set_xticks(range(len(failure_counts)))
ax.set_xticklabels(failure_counts.index, rotation=45, ha='right')
ax.set_ylabel('Count (site-days)')
ax.set_title('Failure Mode Distribution')

# Add percentages on bars
total = len(data)
for i, (bar, count) in enumerate(zip(bars, failure_counts.values)):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count/total*100:.1f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('failure_mode_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Site-Level Analysis - Chronic Problems

In [ ]:
# Aggregate by site to find chronic issues
site_summary = data.groupby('SITEID').agg({
    'SITENAME': 'first',
    'STATE': 'first',
    'IRRADIANCE_TYPE': 'first',
    'SIZE_KW_DC': 'first',
    'DATE': 'count',
    'ONSITE_ZERO': 'sum',
    'SATELLITE_ZERO': 'sum',
    'RATIO_TOO_LOW': 'sum',
    'RATIO_TOO_HIGH': 'sum',
    'RATIO_REASONABLE': 'sum',
    'WA_PR_HIGH': 'sum',
    'WA_PR_LOW': 'sum',
    'ONSITE_SAT_RATIO': ['mean', 'std'],
    'VIEW_WA_PR': ['mean', 'std']
}).reset_index()

# Flatten column names
site_summary.columns = ['_'.join(col).strip('_') if isinstance(col, tuple) else col 
                        for col in site_summary.columns]

site_summary = site_summary.rename(columns={
    'DATE_count': 'DAYS',
    'ONSITE_ZERO_sum': 'DAYS_ONSITE_ZERO',
    'SATELLITE_ZERO_sum': 'DAYS_SATELLITE_ZERO',
    'RATIO_TOO_LOW_sum': 'DAYS_RATIO_LOW',
    'RATIO_TOO_HIGH_sum': 'DAYS_RATIO_HIGH',
    'RATIO_REASONABLE_sum': 'DAYS_RATIO_OK',
    'WA_PR_HIGH_sum': 'DAYS_WA_PR_HIGH',
    'WA_PR_LOW_sum': 'DAYS_WA_PR_LOW',
    'ONSITE_SAT_RATIO_mean': 'AVG_RATIO',
    'ONSITE_SAT_RATIO_std': 'STD_RATIO',
    'VIEW_WA_PR_mean': 'AVG_WA_PR',
    'VIEW_WA_PR_std': 'STD_WA_PR'
})

# Calculate percentages
site_summary['PCT_ONSITE_ZERO'] = (site_summary['DAYS_ONSITE_ZERO'] / site_summary['DAYS'] * 100).round(1)
site_summary['PCT_RATIO_PROBLEM'] = ((site_summary['DAYS_RATIO_LOW'] + site_summary['DAYS_RATIO_HIGH']) / site_summary['DAYS'] * 100).round(1)
site_summary['PCT_WA_PR_HIGH'] = (site_summary['DAYS_WA_PR_HIGH'] / site_summary['DAYS'] * 100).round(1)

print(f"\nSites analyzed: {len(site_summary)}")

In [ ]:
# Sites with chronic onsite sensor issues (>30% zero readings)
chronic_onsite_dead = site_summary[site_summary['PCT_ONSITE_ZERO'] > 30].sort_values('PCT_ONSITE_ZERO', ascending=False)
print("\n" + "=" * 60)
print("SITES WITH CHRONIC ONSITE SENSOR ISSUES (>30% zero readings)")
print("=" * 60)
print(f"Count: {len(chronic_onsite_dead)} sites")
print(chronic_onsite_dead[['SITEID', 'SITENAME_first', 'STATE_first', 'DAYS', 'PCT_ONSITE_ZERO', 'AVG_WA_PR']].head(20))

In [ ]:
# Sites with high WA PR issues (>20% of days with WA PR >120%)
high_wa_pr_sites = site_summary[site_summary['PCT_WA_PR_HIGH'] > 20].sort_values('PCT_WA_PR_HIGH', ascending=False)
print("\n" + "=" * 60)
print("SITES WITH FREQUENT HIGH WA PR (>20% of days with WA PR >120%)")
print("=" * 60)
print(f"Count: {len(high_wa_pr_sites)} sites")
print(high_wa_pr_sites[['SITEID', 'SITENAME_first', 'STATE_first', 'DAYS', 'PCT_WA_PR_HIGH', 'AVG_RATIO', 'AVG_WA_PR']].head(20))

In [ ]:
# Sites with high ratio variability (sensor inconsistency)
high_variability = site_summary[site_summary['STD_RATIO'] > 1.0].sort_values('STD_RATIO', ascending=False)
print("\n" + "=" * 60)
print("SITES WITH HIGH RATIO VARIABILITY (std > 1.0)")
print("=" * 60)
print(f"Count: {len(high_variability)} sites")
print(high_variability[['SITEID', 'SITENAME_first', 'STATE_first', 'DAYS', 'AVG_RATIO', 'STD_RATIO', 'AVG_WA_PR']].head(20))

### 3.4 Regional/Seasonal Patterns

In [ ]:
# Add time features
data['MONTH'] = data['DATE'].dt.month
data['WEEK'] = data['DATE'].dt.isocalendar().week
data['DOW'] = data['DATE'].dt.dayofweek

# Regional analysis
regional = data.groupby('STATE').agg({
    'SITEID': 'nunique',
    'DATE': 'count',
    'ONSITE_SAT_RATIO': 'mean',
    'VIEW_WA_PR': 'mean',
    'RATIO_TOO_LOW': 'mean',
    'RATIO_TOO_HIGH': 'mean',
    'WA_PR_HIGH': 'mean'
}).round(3)

regional.columns = ['Sites', 'Days', 'Avg_Ratio', 'Avg_WA_PR', 'Pct_Ratio_Low', 'Pct_Ratio_High', 'Pct_WA_PR_High']
regional = regional.sort_values('Pct_WA_PR_High', ascending=False)

print("\n" + "=" * 60)
print("REGIONAL PATTERNS")
print("=" * 60)
print(regional.head(15))

In [ ]:
# Monthly patterns
monthly = data.groupby('MONTH').agg({
    'SITEID': 'count',
    'ONSITE_SAT_RATIO': ['mean', 'std'],
    'VIEW_WA_PR': ['mean', 'std'],
    'RATIO_TOO_LOW': 'mean',
    'RATIO_TOO_HIGH': 'mean',
    'WA_PR_HIGH': 'mean'
}).round(3)

print("\n" + "=" * 60)
print("MONTHLY PATTERNS")
print("=" * 60)
print(monthly)

In [ ]:
# Visualize monthly trends
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

monthly_data = data.groupby('MONTH').agg({
    'ONSITE_SAT_RATIO': 'mean',
    'VIEW_WA_PR': 'mean',
    'WA_PR_HIGH': 'mean',
    'ONSITE_ZERO': 'mean'
}).reset_index()

ax1 = axes[0, 0]
ax1.plot(monthly_data['MONTH'], monthly_data['ONSITE_SAT_RATIO'], 'o-', color='steelblue')
ax1.axhline(1.0, color='gray', linestyle='--')
ax1.set_xlabel('Month')
ax1.set_ylabel('Average Onsite/Satellite Ratio')
ax1.set_title('Average Onsite/Satellite Ratio by Month')

ax2 = axes[0, 1]
ax2.plot(monthly_data['MONTH'], monthly_data['VIEW_WA_PR'], 'o-', color='green')
ax2.axhline(1.0, color='gray', linestyle='--')
ax2.set_xlabel('Month')
ax2.set_ylabel('Average WA PR')
ax2.set_title('Average WA PR by Month')

ax3 = axes[1, 0]
ax3.bar(monthly_data['MONTH'], monthly_data['WA_PR_HIGH'] * 100, color='coral')
ax3.set_xlabel('Month')
ax3.set_ylabel('% of Days with WA PR > 120%')
ax3.set_title('Percentage of High WA PR Days by Month')

ax4 = axes[1, 1]
ax4.bar(monthly_data['MONTH'], monthly_data['ONSITE_ZERO'] * 100, color='darkred')
ax4.set_xlabel('Month')
ax4.set_ylabel('% of Days with Onsite = 0')
ax4.set_title('Percentage of Days with Dead Onsite Sensor by Month')

plt.tight_layout()
plt.savefig('monthly_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Investigate Specific Problem Sites

Look at known problem sites mentioned in the context.

In [ ]:
# Check for specific problem sites mentioned
problem_sites = ['SP-110', 'SP-437']
available_problem_sites = [s for s in problem_sites if s in data['SITEID'].values]

print(f"Problem sites found in data: {available_problem_sites}")

if available_problem_sites:
    for site_id in available_problem_sites:
        site_data = data[data['SITEID'] == site_id].copy()
        print(f"\n{'='*60}")
        print(f"SITE: {site_id}")
        print(f"{'='*60}")
        print(f"Days: {len(site_data)}")
        print(f"State: {site_data['STATE'].iloc[0]}")
        print(f"Irradiance Type: {site_data['IRRADIANCE_TYPE'].iloc[0]} ({'GHI' if site_data['IRRADIANCE_TYPE'].iloc[0] == 1 else 'POA'})")
        print(f"\nInsolation Sources:")
        print(f"  POA onsite: min={site_data['ONSITE_POA'].min():.1f}, max={site_data['ONSITE_POA'].max():.1f}, mean={site_data['ONSITE_POA'].mean():.1f}")
        print(f"  GHI onsite: min={site_data['ONSITE_GHI'].min():.1f}, max={site_data['ONSITE_GHI'].max():.1f}, mean={site_data['ONSITE_GHI'].mean():.1f}")
        print(f"  Satellite GHI: min={site_data['SATELLITE_GHI'].min():.1f}, max={site_data['SATELLITE_GHI'].max():.1f}, mean={site_data['SATELLITE_GHI'].mean():.1f}")
        print(f"\nRatio (onsite/satellite):")
        print(f"  Mean: {site_data['ONSITE_SAT_RATIO'].mean():.3f}")
        print(f"  Std: {site_data['ONSITE_SAT_RATIO'].std():.3f}")
        print(f"\nView Selection:")
        print(site_data['VIEW_SOURCE'].value_counts())
        print(f"\nWA PR stats:")
        print(f"  Mean: {site_data['VIEW_WA_PR'].mean():.3f}")
        print(f"  Days >120%: {(site_data['VIEW_WA_PR'] > 1.2).sum()}")
        print(f"\nFailure Modes:")
        print(site_data['FAILURE_MODE'].value_counts())

In [ ]:
# Find worst sites by each failure mode
print("\n" + "=" * 60)
print("TOP 10 SITES BY EACH PROBLEM TYPE")
print("=" * 60)

# Sites with most onsite sensor failures
print("\n--- Sites with Most Onsite Sensor Failures ---")
onsite_failures = data[data['FAILURE_MODE'].isin(['ONSITE_DEAD', 'ONSITE_STUCK_LOW'])].groupby('SITEID').size().sort_values(ascending=False)
print(onsite_failures.head(10))

# Sites with most satellite misses
print("\n--- Sites with Most Satellite Misses ---")
satellite_misses = data[data['FAILURE_MODE'].isin(['SATELLITE_LOW_MISS', 'SATELLITE_HIGH_MISS'])].groupby('SITEID').size().sort_values(ascending=False)
print(satellite_misses.head(10))

# Sites with most high WA PR
print("\n--- Sites with Most High WA PR Days ---")
high_wa_pr = data[data['VIEW_WA_PR'] > 1.2].groupby('SITEID').size().sort_values(ascending=False)
print(high_wa_pr.head(10))

## 5. Simulate Improved Selection Logic

Test different selection strategies and compare results.

In [ ]:
def select_insolation_v2(row):
    """
    Improved insolation selection logic.
    
    Enhancements:
    1. Stricter minimum threshold (200 vs 100)
    2. Narrower valid ratio range (0.5-3.0 vs 0.2-5.0)
    3. Use expected insolation more heavily
    4. Consider consistency over time (rolling window if available)
    5. POA should be 10-30% higher than GHI typically
    """
    MIN_THRESHOLD = 200  # Stricter minimum
    RATIO_MIN = 0.5      # Tighter lower bound
    RATIO_MAX = 3.0      # Tighter upper bound
    EXPECTED_MIN_PCT = 0.10  # Must be at least 10% of expected
    
    onsite_poa = float(row.get('ONSITE_POA', 0) or 0)
    onsite_ghi = float(row.get('ONSITE_GHI', 0) or 0)
    satellite_ghi = float(row.get('SATELLITE_GHI', 0) or 0)
    expected_poa = float(row.get('EXPECTED_INSOLATION_POA', 0) or 0)
    expected_ghi = float(row.get('EXPECTED_INSOLATION_GHI', 0) or 0)
    irradiance_type = int(row.get('IRRADIANCE_TYPE', 2) or 2)
    
    # Determine primary source based on site config
    if irradiance_type == 2:  # POA preferred
        primary_onsite = onsite_poa
        primary_expected = expected_poa
        secondary_onsite = onsite_ghi
        secondary_expected = expected_ghi
    else:  # GHI preferred
        primary_onsite = onsite_ghi
        primary_expected = expected_ghi
        secondary_onsite = onsite_poa
        secondary_expected = expected_poa
    
    def is_valid(onsite, expected, satellite):
        # Must exceed minimum
        if onsite < MIN_THRESHOLD:
            return False, 'below_min'
        
        # Check against expected
        if expected > 0:
            expected_ratio = onsite / expected
            if expected_ratio < EXPECTED_MIN_PCT:
                return False, 'below_expected'
            if expected_ratio > 2.0:
                return False, 'above_expected'
        
        # Cross-check with satellite
        if satellite > MIN_THRESHOLD:
            sat_ratio = onsite / satellite
            if sat_ratio < RATIO_MIN or sat_ratio > RATIO_MAX:
                return False, 'satellite_mismatch'
        
        return True, 'ok'
    
    # Try primary onsite
    valid, reason = is_valid(primary_onsite, primary_expected, satellite_ghi)
    if valid:
        source = 'POA' if irradiance_type == 2 else 'GHI'
        return primary_onsite, source, True, reason
    
    # Try secondary onsite
    valid, reason = is_valid(secondary_onsite, secondary_expected, satellite_ghi)
    if valid:
        source = 'GHI_FALLBACK' if irradiance_type == 2 else 'POA_FALLBACK'
        return secondary_onsite, source, True, reason
    
    # Use satellite as fallback
    if satellite_ghi > MIN_THRESHOLD:
        # For POA sites, adjust satellite GHI upward by ~15% to estimate POA
        if irradiance_type == 2:
            adjusted_satellite = satellite_ghi * 1.15
            return adjusted_satellite, 'SATELLITE_POA_EST', True, 'satellite_adjusted'
        return satellite_ghi, 'SATELLITE_GHI', True, 'satellite_direct'
    
    return max(primary_onsite, satellite_ghi, 0), 'NONE', False, 'no_valid_data'

# Apply new logic
results = data.apply(select_insolation_v2, axis=1, result_type='expand')
data['NEW_INSOLATION'] = results[0]
data['NEW_SOURCE'] = results[1]
data['NEW_VALID'] = results[2]
data['NEW_REASON'] = results[3]

print("New source distribution:")
print(data['NEW_SOURCE'].value_counts())

In [ ]:
# Compute new WA PR with new insolation
data['NEW_GAP'] = (data['NEW_INSOLATION'] / data['EXPECTED_INSOL'].replace(0, np.nan)) - 1
data['NEW_WA_EXPECTED'] = data['EXPECTED_PRODUCTION'] * (1 + data['NEW_GAP'])
data['NEW_WA_PR'] = data['PRODUCTION'] / data['NEW_WA_EXPECTED'].replace(0, np.nan)

# Compare old vs new
comparison = data[['VIEW_WA_PR', 'NEW_WA_PR', 'VIEW_SOURCE', 'NEW_SOURCE']].copy()
comparison['WA_PR_DIFF'] = comparison['NEW_WA_PR'] - comparison['VIEW_WA_PR']
comparison['SOURCE_CHANGED'] = comparison['VIEW_SOURCE'] != comparison['NEW_SOURCE']

print("\n" + "=" * 60)
print("COMPARISON: OLD vs NEW SELECTION LOGIC")
print("=" * 60)
print(f"\nRows where source changed: {comparison['SOURCE_CHANGED'].sum():,} ({comparison['SOURCE_CHANGED'].mean()*100:.1f}%)")
print(f"\nOLD WA PR distribution:")
print(comparison['VIEW_WA_PR'].describe())
print(f"\nNEW WA PR distribution:")
print(comparison['NEW_WA_PR'].describe())

In [ ]:
# Check improvement in problem metrics
old_high_wa_pr = (data['VIEW_WA_PR'] > 1.2).sum()
new_high_wa_pr = (data['NEW_WA_PR'] > 1.2).sum()
old_low_wa_pr = (data['VIEW_WA_PR'] < 0.5).sum()
new_low_wa_pr = (data['NEW_WA_PR'] < 0.5).sum()

print("\n" + "=" * 60)
print("IMPROVEMENT IN PROBLEM METRICS")
print("=" * 60)
print(f"\nWA PR > 120%:")
print(f"  Old: {old_high_wa_pr:,} ({old_high_wa_pr/len(data)*100:.2f}%)")
print(f"  New: {new_high_wa_pr:,} ({new_high_wa_pr/len(data)*100:.2f}%)")
print(f"  Reduction: {old_high_wa_pr - new_high_wa_pr:,} ({(old_high_wa_pr - new_high_wa_pr)/old_high_wa_pr*100:.1f}% improvement)" if old_high_wa_pr > 0 else "")

print(f"\nWA PR < 50%:")
print(f"  Old: {old_low_wa_pr:,} ({old_low_wa_pr/len(data)*100:.2f}%)")
print(f"  New: {new_low_wa_pr:,} ({new_low_wa_pr/len(data)*100:.2f}%)")

In [ ]:
# Visualize WA PR distributions comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Old WA PR
ax1 = axes[0]
valid_old = data['VIEW_WA_PR'].dropna()
valid_old = valid_old[(valid_old > 0) & (valid_old < 3)]
valid_old.hist(bins=50, ax=ax1, alpha=0.7, color='steelblue')
ax1.axvline(1.0, color='green', linestyle='--', linewidth=2, label='Ideal (100%)')
ax1.axvline(1.2, color='red', linestyle='--', linewidth=1, label='High threshold (120%)')
ax1.axvline(0.5, color='orange', linestyle='--', linewidth=1, label='Low threshold (50%)')
ax1.set_xlabel('WA Performance Ratio')
ax1.set_ylabel('Frequency')
ax1.set_title(f'OLD Selection Logic - WA PR Distribution')
ax1.legend()

# New WA PR
ax2 = axes[1]
valid_new = data['NEW_WA_PR'].dropna()
valid_new = valid_new[(valid_new > 0) & (valid_new < 3)]
valid_new.hist(bins=50, ax=ax2, alpha=0.7, color='forestgreen')
ax2.axvline(1.0, color='green', linestyle='--', linewidth=2, label='Ideal (100%)')
ax2.axvline(1.2, color='red', linestyle='--', linewidth=1, label='High threshold (120%)')
ax2.axvline(0.5, color='orange', linestyle='--', linewidth=1, label='Low threshold (50%)')
ax2.set_xlabel('WA Performance Ratio')
ax2.set_ylabel('Frequency')
ax2.set_title(f'NEW Selection Logic - WA PR Distribution')
ax2.legend()

plt.tight_layout()
plt.savefig('wa_pr_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Develop Recommended Thresholds

Based on analysis, propose optimal thresholds.

In [ ]:
# Analyze what ratio ranges correlate with good WA PR
valid_data = data[~data['ONSITE_ZERO'] & ~data['SATELLITE_ZERO'] & data['VIEW_WA_PR'].notna()].copy()

# Bin ratios and look at WA PR distribution
ratio_bins = [0, 0.2, 0.5, 0.7, 0.9, 1.0, 1.1, 1.3, 1.5, 2.0, 3.0, 5.0, 100]
valid_data['RATIO_BIN'] = pd.cut(valid_data['ONSITE_SAT_RATIO'], bins=ratio_bins)

ratio_wa_pr = valid_data.groupby('RATIO_BIN').agg({
    'VIEW_WA_PR': ['mean', 'std', 'count'],
    'WA_PR_HIGH': 'mean',
    'WA_PR_LOW': 'mean'
}).round(3)

print("\n" + "=" * 60)
print("WA PR BY ONSITE/SATELLITE RATIO BIN")
print("=" * 60)
print(ratio_wa_pr)

In [ ]:
# Find the optimal ratio range that minimizes WA PR problems
print("\n" + "=" * 60)
print("RECOMMENDED THRESHOLDS")
print("=" * 60)
print("""
Based on the analysis:

1. MINIMUM INSOLATION THRESHOLD
   Current: 100 Wh/m2/day
   Recommended: 200 Wh/m2/day
   Rationale: Values below 200 are rare in sunny days and often indicate sensor issues

2. ONSITE/SATELLITE RATIO BOUNDS
   Current: 0.2 - 5.0
   Recommended: 0.5 - 2.5 (or 0.6 - 2.0 for stricter)
   Rationale: 
   - Ratios < 0.5 strongly correlate with low WA PR (onsite sensor dead/dirty)
   - Ratios > 2.5 strongly correlate with high WA PR (onsite sensor error or POA vs GHI confusion)
   - POA should be 10-30% higher than GHI, so POA/satellite_GHI ratio of 1.3 is normal

3. EXPECTED INSOLATION CHECK
   Current: > 5% of expected
   Recommended: > 20% of expected AND < 150% of expected
   Rationale: Catches both low and high sensor errors

4. POA vs GHI ADJUSTMENT
   When using satellite GHI for a POA site, multiply by 1.15 to estimate POA
   This prevents systematic underestimation of expected production

5. ROLLING WINDOW VALIDATION (Future Enhancement)
   Flag sensors with high day-to-day variance (std > 1.5) as suspicious
   Detect stuck sensors: same value +/- 5% for 3+ consecutive days
""")

## 7. Export Problem Sites for Review

In [ ]:
# Create summary of problem sites for operations team
problem_site_summary = site_summary[
    (site_summary['PCT_ONSITE_ZERO'] > 20) | 
    (site_summary['PCT_WA_PR_HIGH'] > 15) |
    (site_summary['AVG_RATIO'] < 0.5) |
    (site_summary['AVG_RATIO'] > 2.0)
].copy()

# Add recommendation
def get_recommendation(row):
    issues = []
    if row['PCT_ONSITE_ZERO'] > 50:
        issues.append('CHECK_SENSOR_CONNECTION')
    elif row['PCT_ONSITE_ZERO'] > 20:
        issues.append('INTERMITTENT_SENSOR')
    
    if row['AVG_RATIO'] < 0.5:
        issues.append('SENSOR_LOW_OUTPUT')
    elif row['AVG_RATIO'] > 2.0:
        issues.append('SENSOR_HIGH_OUTPUT_OR_WRONG_TYPE')
    
    if row['PCT_WA_PR_HIGH'] > 30:
        issues.append('USE_SATELLITE_DATA')
    
    return ', '.join(issues) if issues else 'MONITOR'

problem_site_summary['RECOMMENDATION'] = problem_site_summary.apply(get_recommendation, axis=1)

print(f"\nProblem sites requiring attention: {len(problem_site_summary)}")
print(problem_site_summary[['SITEID', 'SITENAME_first', 'STATE_first', 'DAYS', 'PCT_ONSITE_ZERO', 'PCT_WA_PR_HIGH', 'AVG_RATIO', 'RECOMMENDATION']].head(30))

In [ ]:
# Save problem sites to CSV
output_file = 'problem_sites_insolation.csv'
problem_site_summary.to_csv(output_file, index=False)
print(f"\nSaved problem sites to {output_file}")

In [ ]:
# Save detailed analysis data
analysis_output = data[['SITEID', 'DATE', 'STATE', 'IRRADIANCE_TYPE',
                        'ONSITE_POA', 'ONSITE_GHI', 'SATELLITE_GHI',
                        'ONSITE_SAT_RATIO', 'VIEW_SOURCE', 'VIEW_WA_PR',
                        'NEW_SOURCE', 'NEW_WA_PR', 'FAILURE_MODE']].copy()

analysis_output.to_csv('insolation_analysis_full.csv', index=False)
print(f"Saved full analysis to insolation_analysis_full.csv ({len(analysis_output):,} rows)")

## 8. Summary & Recommendations

In [ ]:
print("="*80)
print("INSOLATION SELECTION RESEARCH SUMMARY")
print("="*80)
print(f"""
ANALYSIS PERIOD: {start_date} to {end_date}
SITES ANALYZED: {data['SITEID'].nunique()}
SITE-DAYS: {len(data):,}

KEY FINDINGS:
-------------
1. ONSITE SENSOR ISSUES:
   - {data['ONSITE_ZERO'].sum():,} site-days ({data['ONSITE_ZERO'].mean()*100:.1f}%) have dead onsite sensors
   - {len(chronic_onsite_dead)} sites have chronic sensor issues (>30% zero readings)

2. ONSITE/SATELLITE AGREEMENT:
   - {data['RATIO_REASONABLE'].sum():,} site-days ({data['RATIO_REASONABLE'].mean()*100:.1f}%) have reasonable agreement (0.5-2.0x)
   - {data['RATIO_TOO_LOW'].sum():,} site-days ({data['RATIO_TOO_LOW'].mean()*100:.1f}%) have onsite << satellite (ratio < 0.2)
   - {data['RATIO_TOO_HIGH'].sum():,} site-days ({data['RATIO_TOO_HIGH'].mean()*100:.1f}%) have onsite >> satellite (ratio > 5.0)

3. WA PR ISSUES:
   - {data['WA_PR_HIGH'].sum():,} site-days ({data['WA_PR_HIGH'].mean()*100:.1f}%) have WA PR > 120%
   - {data['WA_PR_LOW'].sum():,} site-days ({data['WA_PR_LOW'].mean()*100:.1f}%) have WA PR < 50%

4. NEW LOGIC IMPACT:
   - Reduces high WA PR days from {old_high_wa_pr:,} to {new_high_wa_pr:,}
   - Source changed for {comparison['SOURCE_CHANGED'].sum():,} site-days ({comparison['SOURCE_CHANGED'].mean()*100:.1f}%)

RECOMMENDATIONS:
----------------
1. Increase minimum threshold from 100 to 200 Wh/m2/day
2. Tighten ratio bounds from 0.2-5.0 to 0.5-2.5
3. Add upper bound check vs expected (< 150%)
4. Adjust satellite GHI by 1.15x for POA sites
5. Implement rolling window for stuck sensor detection
6. Flag {len(problem_site_summary)} sites for sensor maintenance
""")

In [ ]:
# Close connection
connector.disconnect()
print("\nSnowflake connection closed.")